In [ ]:
import asyncio
import time
from collections import abc

from glow import timer, time_this

DT = 0.1
FAIL = False


def _cpu_wa1_fr1_sus0_cpu1_io0():  # wall=1, frame=1, susp=0, cpu=1, io=0 (wall=1, process=1, thread=1)
    # busy += dt
    end = time.perf_counter() + DT
    while time.perf_counter() < end:
        sum(range(1000))


def _sleep_wa1_fr1_sus0_cpu0_io1():  # wall=1, frame=1, susp=0, cpu=0, io=1
    # idle += dt
    time.sleep(DT)


async def _asleep_wa1_fr0_sus1_cpu0_io1():  # wall=1, frame=0, susp=1, cpu=0, io=1
    # idle += dt
    await asyncio.sleep(DT)


def work_wa2_fr2_sus0_cpu1_io1():  # wall=2, frame=2, susp=0, cpu=1, io=1
    _cpu_wa1_fr1_sus0_cpu1_io0()
    _sleep_wa1_fr1_sus0_cpu0_io1()
    if FAIL:
        1 / 0


async def awork_wa3_fr2_sus1_cpu1_io2():  # wall=3, frame=2, susp=1, cpu=1, io=2
    _cpu_wa1_fr1_sus0_cpu1_io0()
    _sleep_wa1_fr1_sus0_cpu0_io1()
    await _asleep_wa1_fr0_sus1_cpu0_io1()
    if FAIL:
        1 / 0


class _Iter660:
    def __init__(self) -> None:
        self.it = iter(range(0, 4, 2))

    def __iter__(self):
        return self

    def __next__(self):  # wall=2, frame=2, cpu=1, io=1
        i = self.it.__next__()  # 0/0/0/0
        work_wa2_fr2_sus0_cpu1_io1()  # 2/2/1/1
        return _SubIter2211(i)  # 0/0/0/0

    def __repr__(self):
        return f'<{self.__class__.__name__} at {id(self):x}>'


class _SubIter2211:
    def __init__(self, i) -> None:
        self.it = iter(range(i, i + 2))

    def __iter__(self):
        return self

    def __next__(self):  # wall=2, frame=2, cpu=1, io=1
        i = self.it.__next__()  # 0/0/0/0
        work_wa2_fr2_sus0_cpu1_io1()  # 2/2/1/1
        return i  # 0/0/0/0

    def __repr__(self):
        return f'<{self.__class__.__name__} at {id(self):x}>'

In [4]:
tps = (
    abc.Iterator,
    abc.Generator,  # is iterator
    abc.Awaitable,
    abc.Coroutine,  # is awaitable
    abc.AsyncIterator,
    abc.AsyncGenerator,  # is async iterator
)


def _info[T](obj: T, tag: str = '') -> T:
    print(type(obj).__qualname__, obj, tag)

    bases = [
        tp.__name__
        for tp in tps
        if (
            issubclass(obj, tp)
            if isinstance(obj, type)
            else isinstance(obj, tp)
        )
    ]
    base = {
        *dir(object()),
        *dir(object),
        '__doc__',
        '__module__',
        '__class_getitem__',
        '__del__',
        '__name__',
        '__qualname__',
        '__dict__',
        '__weakref__',
    }
    if bases:
        print('- bases:', *bases)

    props = {k: callable(getattr(obj, k)) for k in sorted({*dir(obj)} - base)}
    if attrs := [k for k, ismethod in props.items() if not ismethod]:
        print('- attrs:', *attrs)
    if methods := [k for k, ismethod in props.items() if ismethod]:
        print('- methods:', *methods)
    return obj

In [31]:
async def adef() -> int:
    try:
        print('<sleep ...>')
        await asyncio.sleep(0.5)
        print('<sleep ok>')
        # async def a2():
        #     await asyncio.sleep(.1)
        #     1 / 0
        #     return 53
        # await asyncio.ensure_future(a2())
    except asyncio.CancelledError:
        print('<sleep err>')
        await asyncio.sleep(0.5)
        print('<sleep 2nd time>')
    except BaseException as e:
        print(f'<sleep base err {e!r}>')
        await asyncio.sleep(0.5)
        print('<sleep 2nd time>')
        raise
    return 42


aw = coro = _info(adef())  # -> coroutine: Coroutine & Awaitable


async def coro2():
    while True:
        try:
            f: asyncio.Future = _info(coro.send(None), '... <- await X')
        except StopIteration as e:
            return _info(e.value, '... <- return X')
        else:
            # _info(f.__await__(), 'X = _.__await__() <- await ...')

            ev = asyncio.Event()
            f.add_done_callback(lambda _, ev=ev: ev.set())

            try:
                await ev.wait()
            except BaseException as e:
                coro.throw(e)
                raise

            # try:
            #     while not f.done():
            #         await asyncio.sleep(0)
            # except BaseException as e:
            #     f.__await__().throw(e)
            #     raise


t = asyncio.create_task(coro2())
await asyncio.sleep(0.1)
t.cancel()

coroutine <coroutine object adef at 0x0000027C7F765380> 
- bases: Awaitable Coroutine
- attrs: cr_await cr_code cr_frame cr_origin cr_running cr_suspended
- methods: __await__ close send throw
<sleep ...>
Future <Future pending> ... <- await X
- bases: Awaitable
- attrs: _asyncio_future_blocking _callbacks _cancel_message _exception _log_traceback _loop _result _source_traceback _state
- methods: __await__ __iter__ _make_cancelled_error add_done_callback cancel cancelled done exception get_loop remove_done_callback result set_exception set_result


True

<sleep err>


In [ ]:
@time_this
def fn_iter770():  # function -> iterator[int]
    work_wa2_fr2_sus0_cpu1_io1()
    return _Iter660()


try:
    r_fn_iter = fn_iter770()
    print(':: called ::')
    print([x for xs in r_fn_iter for x in xs])
finally:
    fn_iter770.log_timing()

 1.15% - 93.8ms +  106ms - __main__.fn_iter770 - 


ZeroDivisionError: division by zero

In [7]:
@time_this
def fn_gen770():  # function -> generator[int]
    work_wa2_fr2_sus0_cpu1_io1()
    return (x for x in _Iter660())


try:
    r_fn_gen = fn_gen770()
    print(':: called ::')
    print([x for xs in r_fn_gen for x in xs])
finally:
    fn_gen770.log_timing()

:: called ::
[0, 1, 2, 3]
49.45% -  703ms +  699ms - __main__.fn_gen770 - 


In [8]:
@time_this
def gen770():  # generator[int]
    work_wa2_fr2_sus0_cpu1_io1()
    yield from _Iter660()


r_gen = gen770()
print(':: called ::')
try:
    print([x for xs in r_gen for x in xs])
finally:
    gen770.log_timing()

:: called ::
[0, 1, 2, 3]
32.09% -  672ms +  732ms - __main__.gen770 - 


In [26]:
@time_this
async def coro111():  # coroutine[int]
    # await _asleep_wa1_fr0_sus1_cpu0_io1()
    # await asyncio.to_thread(_cpu_wa1_fr1_sus0_cpu1_io0)
    await awork_wa3_fr2_sus1_cpu1_io2()
    return 42


r_coro = coro111()
print(':: called ::')
try:
    _info(await r_coro)
finally:
    coro111.log_timing()

:: called ::
int 42 
- attrs: denominator imag numerator real
- methods: __abs__ __add__ __and__ __bool__ __ceil__ __divmod__ __float__ __floor__ __floordiv__ __getnewargs__ __index__ __int__ __invert__ __lshift__ __mod__ __mul__ __neg__ __or__ __pos__ __pow__ __radd__ __rand__ __rdivmod__ __rfloordiv__ __rlshift__ __rmod__ __rmul__ __ror__ __round__ __rpow__ __rrshift__ __rshift__ __rsub__ __rtruediv__ __rxor__ __sub__ __truediv__ __trunc__ __xor__ as_integer_ratio bit_count bit_length conjugate from_bytes is_integer to_bytes
 1.57% - 93.8ms +  207ms - __main__.coro111 - 


In [27]:
@time_this
async def coro_iter771():  # coroutine[iterator[int]]
    await awork_wa3_fr2_sus1_cpu1_io2()
    await asyncio.ensure_future(asyncio.sleep(0.1))
    return _Iter660()


r_coro_iter = coro_iter771()
print(':: called ::')
try:
    _info(aw := await r_coro_iter)
    print(':: awaited ::')
    print([x for xs in aw for x in xs])
finally:
    coro_iter771.log_timing()

:: called ::
_Iter660 <_Iter660 at 27c7f200470> 
- bases: Iterator
- attrs: it
- methods: __iter__ __next__
:: awaited ::
[0, 1, 2, 3]
 1.41% - 93.8ms +  324ms - __main__.coro_iter771 - 


In [11]:
@time_this
async def coro_gen771():  # coroutine[generator[int]]
    await awork_wa3_fr2_sus1_cpu1_io2()
    return (x for x in _Iter660())


r_coro_gen = coro_gen771()
print(':: called ::')
try:
    _info(aw := await r_coro_gen)
    print(':: awaited ::')
    print([x for xs in aw for x in xs])
finally:
    coro_gen771.log_timing()

:: called ::
generator <generator object coro_gen771.<locals>.<genexpr> at 0x0000027C7F720640> 
- bases: Iterator Generator
- attrs: gi_code gi_frame gi_running gi_suspended gi_yieldfrom
- methods: __iter__ __next__ close send throw
:: awaited ::
[0, 1, 2, 3]
 2.54% - 93.8ms +  220ms - __main__.coro_gen771 - 


In [12]:
@time_this
async def asyncgen0():  # async generator[int]
    await asyncio.sleep(0.1)
    yield 0
    time.sleep(0.1)
    yield 1
    # raise StopIteration(2)  # ! RuntimeError
    # raise StopAsyncIteration(2)  # ! RuntimeError
    # raise GeneratorExit  # ! ok


try:
    r_asyncgen = asyncgen0()
    print(':: called ::')
    print([x async for x in r_asyncgen])
finally:
    asyncgen0.log_timing()

:: called ::
[0, 1]
 0.00% -     0s +  213ms - __main__.asyncgen0 - 


In [13]:
@time_this
async def asyncgen0():  # async generator[int]
    yield 0
    yield 1
    # return None  # ! SyntaxError
    # raise StopIteration(2)  # ! RuntimeError
    # raise StopAsyncIteration(2)  # ! RuntimeError
    # raise GeneratorExit  # ! ok


try:
    r_asyncgen = asyncgen0()
    print(':: called ::')
    print([x async for x in r_asyncgen])
finally:
    asyncgen0.log_timing()

:: called ::
[0, 1]
 0.00% -     0s +  9.2us - __main__.asyncgen0 - 


In [29]:
@time_this
async def asyncgen771():  # async generator[int]
    await awork_wa3_fr2_sus1_cpu1_io2()
    await asyncio.gather(
        asyncio.ensure_future(asyncio.sleep(DT)),
        asyncio.sleep(0),
    )
    for x in _Iter660():
        yield x


r_asyncgen = asyncgen771()
print(':: called ::')
try:
    print([x async for xs in r_asyncgen for x in xs])
finally:
    asyncgen771.log_timing()

:: called ::
[0, 1, 2, 3]
 3.87% -  312ms +  515ms - __main__.asyncgen771 - 


In [15]:
for obj in (
    r_fn_iter,
    r_fn_gen,
    r_gen,
    r_coro,
    r_coro_iter,
    r_coro_gen,
    r_asyncgen,
):
    print(obj, type(obj))
    # print(type(o).mro())
    # print(sorted({*dir(o)} - {*dir(object())} - {*dir(object)}
    #              - {'__del__', '__name__', '__qualname__'}))
    print('  mro:', [
        tp.__name__ for tp in (
            abc.Iterator,
            abc.Generator,  # is iterator
            abc.Awaitable,
            abc.Coroutine,  # is awaitable
            abc.AsyncIterator,
            abc.AsyncGenerator,  # is async iterator
        ) if isinstance(obj, tp)
    ])

<_Iter660 at 27c7f1f7680> <class 'glow._wrap._Iterator'>
  mro: ['Iterator']
<generator object fn_gen770.<locals>.<genexpr> at 0x0000027C7F720100> <class 'glow._wrap._Generator'>
  mro: ['Iterator', 'Generator']
<generator object gen770 at 0x0000027C7F003C40> <class 'glow._wrap._Generator'>
  mro: ['Iterator', 'Generator']
<coroutine object coro111 at 0x0000027C7F720580> <class 'glow._wrap._Coroutine'>
  mro: ['Awaitable', 'Coroutine']
<coroutine object coro_iter771 at 0x0000027C7F718380> <class 'glow._wrap._Coroutine'>
  mro: ['Awaitable', 'Coroutine']
<coroutine object coro_gen771 at 0x0000027C7F720400> <class 'glow._wrap._Coroutine'>
  mro: ['Awaitable', 'Coroutine']
<async_generator object asyncgen771 at 0x0000027C7F6CFAE0> <class 'glow._wrap._AsyncGenerator'>
  mro: ['AsyncIterator', 'AsyncGenerator']


In [16]:
import asyncio
from glow import memoize

@memoize(3, batched=True)
async def func(xs):
    await asyncio.sleep(0.2)
    raise RuntimeError(xs)

async def sleepy(x, t):
    await asyncio.sleep(t)
    return await func([x])

async def catch():
    # await func([5, 5])
    await asyncio.gather(sleepy(5, 0.2), sleepy(5, 0.1))

await catch()

RuntimeError: [5]